# NB_CursorUsage_Bronze_To_Silver
**Source:** `AI_Medallion.Bronze.Cursor_CursorUsage`  
**Targets (reads):** `OktaUserforAI`, `HC_Silver_Historical`  
**Target:** `AI_Medallion.Silver.CursorUsage`  
**Run after:** NB_OktaUserforAI_Bronze_To_Silver, NB_HCHistorical_Bronze_To_Silver

In [ ]:
%run ./common/common_bootstrap


In [ ]:
bronze_df = spark.table(TABLE_CURSOR_BRONZE)
if CURSOR_BRONZE_FILENAME:
    bronze_df = bronze_df.filter(F.col("filename") == CURSOR_BRONZE_FILENAME)
bronze = bronze_df

okta = spark.table(TABLE_OKTA_SILVER)
hc = spark.table(TABLE_HC_SILVER)

user_norm = F.lower(F.trim(F.col("User")))



In [10]:
base = (
    bronze.filter(is_valid_email(F.col("User")))
    .select(
        F.col("Date").cast("date").alias("CursorUsageDate"),
        user_norm.alias("CursorUsageUser"),
        F.col("`Service Account Name`").alias("CursorUsageServiceAccountName"),
        F.col("`Service Account ID`").alias("CursorUsageServiceAccountID"),
        F.col("`Cloud Agent ID`").alias("CursorUsageCloudAgentID"),
        F.col("`Automation ID`").alias("CursorUsageAutomationID"),
        F.col("Kind").alias("CursorUsageKind"),
        F.col("Model").alias("CursorUsageModel"),
        F.col("`Max Mode`").alias("CursorUsageMaxMode"),
        F.col("`Input (w/ Cache Write)`").cast("long").alias("CursorUsageInputWithCacheWrite"),
        F.col("`Input (w/o Cache Write)`").cast("long").alias("CursorUsageInputWithoutCacheWrite"),
        F.col("`Cache Read`").cast("long").alias("CursorUsageCacheRead"),
        F.col("`Output Tokens`").cast("long").alias("CursorUsageOutputTokens"),
        F.col("`Total Tokens`").cast("long").alias("CursorUsageTotalTokens"),
        F.col("Cost").cast("decimal(18,8)").alias("CursorUsageCost"),
        F.col("FileName").alias("CursorUsageFileName"),
        F.col("ExecutionDate").cast("timestamp").alias("CursorUsageIngestionDate"),
    )
    .withColumn(
        "CursorUsageMonthNo",
        (F.year("CursorUsageDate") * F.lit(100) + F.month("CursorUsageDate")).cast("int"),
    )
)

okta_slim = okta.select(
    "OktaUserId",
    "OktaUserEmail",
    "OktaUserFullEmail",
    "OktaUserEmployeeNumber",
    "OktaUserFullName",
    "OktaUserRegion",
    "OktaUserCountry",
)

# Inner join: match login or full email
joined = base.join(
    okta_slim,
    (F.col("CursorUsageUser") == F.col("OktaUserEmail"))
    | (F.col("CursorUsageUser") == F.col("OktaUserFullEmail")),
    "inner",
)

joined = joined.withColumn(
    "CursorUsageKeyEmail",
    F.when(
        F.col("OktaUserEmail").isNotNull(),
        F.concat(
            F.col("CursorUsageMonthNo").cast("string"),
            F.lit("/"),
            F.col("OktaUserEmail"),
        ),
    ),
).withColumn(
    "CursorUsageKeyFullEmail",
    F.when(
        F.col("OktaUserFullEmail").isNotNull(),
        F.concat(
            F.col("CursorUsageMonthNo").cast("string"),
            F.lit("/"),
            F.col("OktaUserFullEmail"),
        ),
    ),
)

hc_slim = hc.select(
    F.col("HCKeyEmail"),
    F.col("HCBusinessUnit"),
    F.col("HCBusinessUnitCode"),
    F.col("HCDivision"),
    F.col("HCJobTitle"),
    F.col("HCOffice"),
    F.col("HCTalentSegment"),
    F.col("Month_No").alias("HCMonthNo"),
    F.col("OktaUserCountry").alias("HCOktaUserCountry"),
    F.col("HCDivisionINC"),
)

joined = joined.join(
    hc_slim,
    (F.col("HCKeyEmail") == F.col("CursorUsageKeyEmail"))
    | (F.col("HCKeyEmail") == F.col("CursorUsageKeyFullEmail")),
    "left",
).withColumn(
    "HCDivisionINC",
    F.coalesce(F.col("HCDivisionINC"), F.lit(HC_DIVISION_INC_DEFAULT)),
)

write_df= joined.distinct()

write_df=write_df.withColumn(
    FACT_CURSOR_ACTIVE_MERGE_KEY,
    cursor_usage_record_key()
    )

StatementMeta(, bb9bb06d-c4e0-40aa-b603-8e642e55f61c, 24, Finished, Available, Finished, False)

In [ ]:
merge_incremental(
    spark,
    write_df,
    TABLE_CURSOR_SILVER,
    FACT_CURSOR_ACTIVE_MERGE_KEY,
)